# 02 · Seria i uderzenia
**Cel:** utrzymać porządek w danych, sprawdzić każdy podział i wyeksportować cechy. Jeden `plate_id` oznacza jeden fizyczny detal — także gdy nagrasz go w kilku plikach.

In [1]:
from pathlib import Path
import os, json, sys
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists(), "Otwórz cały folder projektu w VS Code."
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
from pinglab.audio import Config, load_audio, detect_hits, cut_hits, listen
from pinglab.session import load_session
SESSION = load_session(ROOT)
CONFIG, FMAX = SESSION.config, SESSION.fmax
MANIFEST, ANNOTATIONS = SESSION.manifest, SESSION.annotations
print("Dane:", SESSION.source_kind, "· manifest:", MANIFEST.relative_to(ROOT) if MANIFEST.is_relative_to(ROOT) else MANIFEST.name)
print("Wspólne pasmo do", FMAX, "Hz · konfiguracja: project.json")
if SESSION.source_kind == "synthetic":
    display(Markdown("**SYMULACJA — wynik sprawdza narzędzia, nie wykrywanie pęknięć u klienta.**"))


Dane: synthetic · manifest: data/demo/manifest.csv
Wspólne pasmo do 20000.0 Hz · konfiguracja: project.json


**SYMULACJA — wynik sprawdza narzędzia, nie wykrywanie pęknięć u klienta.**

In [2]:
# KONFIGURACJA
MANIFEST = SESSION.manifest
ANNOTATIONS = SESSION.annotations
OUT = SESSION.output / "batch"


## Manifest
Wiersz = plik. Obowiązkowe: `path`, `plate_id`, `label`, `session_id`. Etykiety: `ok`, `nok`, `unknown`, `background`. Ścieżka względna jest liczona **od pliku CSV**. Szablon: `data/manifest.template.csv`.

`channel` = opcjonalny numer kanału (0/1, puste = auto). `position` = miejsce uderzenia; `environment` = warunki; `notes` = opis wady/podparcia; `photo` = ścieżka zdjęcia; `plate_type` = typ płyty; `temperature` = temperatura. Przy wielu miejscach w jednym pliku wpisz kolejność i zachowuj ten sam protokół.

In [3]:
from pinglab.dataset import read_manifest, analyze_manifest
manifest = read_manifest(MANIFEST)
display(manifest.drop(columns=['resolved_path','sha256']).head(12))
print("Fizyczne płytki:",manifest[manifest.label!='background'].groupby('label').plate_id.nunique().to_dict())

,path,plate_id,label,session_id,position,environment,notes,photo,channel,plate_type,temperature
0,SYN_001_ok.wav,SYN_001,ok,S1,A/B/C,synthetic,"SYMULACJA, nie ceramika klienta",,,,
1,SYN_002_ok.wav,SYN_002,ok,S2,A/B/C,synthetic,"SYMULACJA, nie ceramika klienta",,,,
2,SYN_003_ok.wav,SYN_003,ok,S3,A/B/C,synthetic,"SYMULACJA, nie ceramika klienta",,,,
3,SYN_004_ok.wav,SYN_004,ok,S1,A/B/C,synthetic,"SYMULACJA, nie ceramika klienta",,,,
4,SYN_005_ok.wav,SYN_005,ok,S2,A/B/C,synthetic,"SYMULACJA, nie ceramika klienta",,,,
5,SYN_006_ok.wav,SYN_006,ok,S3,A/B/C,synthetic,"SYMULACJA, nie ceramika klienta",,,,
6,SYN_007_ok.wav,SYN_007,ok,S1,A/B/C,synthetic,"SYMULACJA, nie ceramika klienta",,,,
7,SYN_008_ok.wav,SYN_008,ok,S2,A/B/C,synthetic,"SYMULACJA, nie ceramika klienta",,,,
8,SYN_009_ok.wav,SYN_009,ok,S3,A/B/C,synthetic,"SYMULACJA, nie ceramika klienta",,,,
9,SYN_010_ok.wav,SYN_010,ok,S1,A/B/C,synthetic,"SYMULACJA, nie ceramika klienta",,,,


Fizyczne płytki: {'nok': 10, 'ok': 20}


## Zatwierdź uderzenia
Wpisz poprawione sekundy do `CORRECTIONS` i uruchom ponownie. Puste `{}` niczego nie zmienia. Opcjonalnie ustaw `SHOW_REVIEW_WIDGET=True`, aby użyć przycisków edytora (VS Code może poprosić o pobranie zasobów). Większy próg wykrywa mniej kandydatów. Zatwierdzone czasy mają pierwszeństwo przed automatem; aby przeliczyć je automatycznie, usuń wpis z JSON. Puste pole i zapis oznaczają świadome odrzucenie wszystkich uderzeń.

In [4]:
# Ręczna korekta bez widgetów: klucz = dokładne path z manifestu, wartości = sekundy.
CORRECTIONS = {}  # np. {"sample_a.wav": [0.42, 1.93, 3.48]}; [] odrzuca plik
if CORRECTIONS:
    from pinglab.audio import cut_hits
    known = set(manifest.path)
    assert set(CORRECTIONS) <= known, "Klucz korekty musi zgadzać się z path w manifeście."
    for path, seconds in CORRECTIONS.items():
        rec = manifest[manifest.path == path].iloc[0]
        recording = load_audio(rec.resolved_path,CONFIG,channel=int(rec.channel) if rec.channel else None)
        cut_hits(recording.y,recording.sr,np.asarray(seconds,float),CONFIG)  # walidacja czasów
    existing = json.loads(ANNOTATIONS.read_text()) if ANNOTATIONS.exists() else {}
    ANNOTATIONS.parent.mkdir(parents=True,exist_ok=True)
    ANNOTATIONS.write_text(json.dumps({**existing,**CORRECTIONS},indent=2))
    print("Zapisano korekty:",list(CORRECTIONS))
SHOW_REVIEW_WIDGET = False  # opcjonalne; podstawowy Run All nie wymaga widgetów
if SHOW_REVIEW_WIDGET:
    from pinglab.widgets import review_manifest
    review = review_manifest(MANIFEST,ANNOTATIONS,CONFIG)

## Przelicz serię
Komórka działa także przed ręcznym przeglądem, ale wynik zaznaczy niezatwierdzone pliki. Przy zmianie okna / pasma przelicz wszystkie pliki. W notebooku ten katalog wyników jest aktualizowany; do zachowania wersji zmień `OUT`.

In [5]:
events, quality = analyze_manifest(MANIFEST,OUT,CONFIG,FMAX,
                                   ANNOTATIONS if ANNOTATIONS.exists() else None)
display(quality[['path','label','peak_dbfs','clipping_fraction','n_hits','reviewed','warnings']])
print("Uderzenia:", len(events), "· Wyniki:", OUT)
assert not events.empty, "Nie wykryto uderzeń; popraw segmentację."
display(events[['plate_id','label','hit_id','start_s','end_s','truncated']].head())

,path,label,peak_dbfs,clipping_fraction,n_hits,reviewed,warnings
0,SYN_001_ok.wav,ok,-3.956122,0.0,3,True,
1,SYN_002_ok.wav,ok,-3.134742,0.0,3,True,
2,SYN_003_ok.wav,ok,-3.305335,0.0,3,True,
3,SYN_004_ok.wav,ok,-3.730169,0.0,3,True,
4,SYN_005_ok.wav,ok,-4.800968,0.0,3,True,
5,SYN_006_ok.wav,ok,-6.851207,0.0,3,True,
6,SYN_007_ok.wav,ok,-3.547069,0.0,3,True,
7,SYN_008_ok.wav,ok,-5.650114,0.0,3,True,
8,SYN_009_ok.wav,ok,-6.536152,0.0,3,True,
9,SYN_010_ok.wav,ok,-5.302261,0.0,3,True,


Uderzenia: 90 · Wyniki: /Users/karolsii/dev/ceramic-ping-lab/reports/runs/demo-session/batch


,plate_id,label,hit_id,start_s,end_s,truncated
0,SYN_001,ok,SYN_001:534ea0236b:1,0.375,1.4,False
1,SYN_001,ok,SYN_001:534ea0236b:2,1.875,2.9,False
2,SYN_001,ok,SYN_001:534ea0236b:3,3.375,4.4,False
3,SYN_002,ok,SYN_002:d53f7c06be:1,0.375,1.4,False
4,SYN_002,ok,SYN_002:d53f7c06be:2,1.875,2.9,False


In [6]:
from pinglab.report import write_report
source_kind = SESSION.source_kind
report = write_report(events,quality,OUT/'report.html',source_kind=source_kind)
display(Markdown(f"Raport HTML: `{report}`"))

Raport HTML: `/Users/karolsii/dev/ceramic-ping-lab/reports/runs/demo-session/batch/report.html`

**Pliki wynikowe:** `quality.csv` — jakość; `features.csv` — cechy każdego uderzenia; `run.json` — parametry i sumy kontrolne. Dalej: **03** — podobieństwa i mały model.